In [0]:
dbutils.widgets.text("catalog", "tpch_dev", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
# ---------------------------------------------------------------------------
# Task 5 – Data Quality & Reconciliation
# Runs after the Gold layer is built.  Raises a pipeline-stopping exception
# (ValueError) whenever any CRITICAL check fails so the job/pipeline aborts.
# WARNING-level checks are logged but never stop the run.
# ---------------------------------------------------------------------------

import logging
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("DQ_RECON")

# =====================================================================
# 0.  Configuration – table identifiers (3-layer catalog.schema.table)
# =====================================================================
DB_BRONZE = f"{catalog}.bronze"
DB_SILVER = f"{catalog}.silver"
DB_GOLD   = f"{catalog}.gold"

# The eight source tables that must have been ingested
EIGHT_SOURCE_TABLES = [
    "customers",            # order-level sales headers
    "lineitem", # line-item detail
    "nation",
    "orders",
    "part",
    "partsupp",
    "region",
    "supplier",
]

# ---------------------------------------------------------------------
# Hard-coded primary / composite keys for every Silver & Gold table
# Single-column PKs use a one-element list; composite PKs list all columns.
# ---------------------------------------------------------------------
TABLE_KEYS = {
    # --- Silver layer (cleaned TPC-H source tables) ---
    f"{DB_SILVER}.customers":          ["c_custkey"],
    f"{DB_SILVER}.lineitem":            ["l_orderkey", "l_linenumber"],
    f"{DB_SILVER}.nation":              ["n_nationkey"],
    f"{DB_SILVER}.orders":              ["o_orderkey"],
    f"{DB_SILVER}.part":                ["p_partkey"],
    f"{DB_SILVER}.partsupp":            ["ps_partkey", "ps_suppkey"],
    f"{DB_SILVER}.region":              ["r_regionkey"],
    f"{DB_SILVER}.supplier":            ["s_suppkey"],
    # --- Gold layer (dimension tables) ---
    f"{DB_GOLD}.dim_customer":          ["customer_key"],
    f"{DB_GOLD}.dim_product":              ["product_key"],
    f"{DB_GOLD}.dim_supplier":          ["supplier_key"],
    f"{DB_GOLD}.dim_geography":            ["geography_key"],
    # --- Gold layer (fact tables – composite keys) ---
    f"{DB_GOLD}.fact_sales":            ["order_key", "line_number"]
}

# ---------------------------------------------------------------------
# Helper utilities
# ---------------------------------------------------------------------
class DQFailure(Exception):
    """Raised when a CRITICAL data-quality check fails – stops the pipeline."""


class DataQualityRunner:
    def __init__(self):
        self.critical_failures = []   # list[str] – messages for critical failures
        self.warnings          = []   # list[str] – messages for warnings

    # -- check primitives --------------------------------------------------
    def critical(self, msg: str):
        log.error(f"CRITICAL: {msg}")
        self.critical_failures.append(msg)

    def warn(self, msg: str):
        log.warning(f"WARNING: {msg}")
        self.warnings.append(msg)

    def assert_table_exists(self, full_name: str, label: str = ""):
        """Return True when the table exists and is queryable."""
        try:
            spark.sql(f"DESCRIBE TABLE {full_name}")
            return True
        except Exception:
            self.critical(f"{label or full_name}: table does not exist.")
            return False

    # -- generic metric helpers -------------------------------------------
    def _scalar(self, sql: str):
        return spark.sql(sql).collect()[0][0]

    def _row_count(self, table_name: str) -> int:
        return self._scalar(f"SELECT COUNT(*) FROM {table_name}")

    # -- dynamic table discovery helpers ----------------------------------
    def _list_tables(self, schema: str) -> list:
        """Return list of table names that actually exist in the given schema."""
        try:
            rows = spark.sql(f"SHOW TABLES IN {schema}").collect()
            return [r["tableName"] for r in rows if not r["isTemporary"]]
        except Exception:
            return []

    def _get_columns(self, table_name: str) -> list:
        """Return list of column names for a table."""
        try:
            rows = spark.sql(f"DESCRIBE TABLE {table_name}").collect()
            cols = []
            for r in rows:
                cn = r["col_name"]
                if not cn or cn.startswith("#"):
                    if cols:
                        break
                    continue
                cols.append(cn)
            return cols
        except Exception:
            return []

    def _infer_pk_column(self, table_name: str, columns: list):
        """Infer the most likely single-column PK from column names."""
        tbl = table_name.split(".")[-1].lower()
        base = tbl.replace("dim_", "").replace("fact_", "")
        # 1. Exact match: {base}_key or {base}_id
        for col in columns:
            if col.lower() in (f"{base}_key", f"{base}_id"):
                return col
        # 2. TPC-H style: {first_letter}_{base}_key
        for col in columns:
            if col.lower() == f"{base[0]}_{base}_key":
                return col
        # 3. Any column ending in _key or _id (first match)
        for col in columns:
            if col.lower().endswith("_key") or col.lower().endswith("_id"):
                return col
        return None

    # ===================================================================
    # 1.  All eight source tables were ingested
    # ===================================================================
    def check_source_ingestion(self):
        log.info("=" * 70)
        log.info("CHECK 1 – All eight source tables ingested into Bronze")
        log.info("=" * 70)
        for tbl in EIGHT_SOURCE_TABLES:
            full = f"{DB_BRONZE}.{tbl}"
            if not self.assert_table_exists(full, tbl):
                continue
            cnt = self._row_count(full)
            if cnt == 0:
                self.critical(f"{full}: ingested but contains 0 rows.")
            else:
                log.info(f"  OK  {full:50s} rows={cnt:,}")

    # ===================================================================
    # 2.  Primary keys are populated and unique
    # ===================================================================
    def check_primary_keys(self):
        log.info("=" * 70)
        log.info("CHECK 2 – Primary keys populated and unique (hard-coded)")
        log.info("=" * 70)

        for table, pk_cols in TABLE_KEYS.items():
            if not self.assert_table_exists(table, table):
                continue

            # Verify all configured PK columns actually exist in the table
            columns = self._get_columns(table)
            missing = [c for c in pk_cols if c not in columns]
            if missing:
                self.warn(f"{table}: configured PK column(s) {missing} not found in schema, skipping.")
                continue

            pk_label = ", ".join(pk_cols)

            # NULL check – any PK column NULL is a failure
            null_conditions = " OR ".join(f"{c} IS NULL" for c in pk_cols)
            null_cnt = self._scalar(
                f"SELECT COUNT(*) FROM {table} WHERE {null_conditions}"
            )
            if null_cnt > 0:
                self.critical(f"{table}: PK ({pk_label}) has {null_cnt:,} row(s) with NULL value(s).")

            # Duplicate check – supports composite keys via multi-column GROUP BY
            group_cols = ", ".join(
                f"COALESCE(CAST({c} AS STRING), '__NULL__')" for c in pk_cols
            )
            dup_cnt = self._scalar(
                f"""
                SELECT COUNT(*) FROM (
                    SELECT {group_cols} AS pk_val, COUNT(*) AS cnt
                    FROM {table}
                    GROUP BY {group_cols}
                    HAVING COUNT(*) > 1
                )
                """
            )
            if dup_cnt > 0:
                self.critical(f"{table}: PK ({pk_label}) has {dup_cnt:,} duplicate value(s).")
            else:
                log.info(f"  OK  {table:50s} PK=({pk_label}) is populated & unique")

    # ===================================================================
    # 3.  Fact table – no null surrogate keys
    # ===================================================================
    def check_fact_surrogate_keys(self):
        log.info("=" * 70)
        log.info("CHECK 3 – Fact table contains no NULL surrogate keys (hard-coded)")
        log.info("=" * 70)

        # Use the fact-table keys from TABLE_KEYS instead of dynamic discovery
        fact_tables = {
            t: cols for t, cols in TABLE_KEYS.items()
            if t.split(".")[-1].lower().startswith("fact_")
        }

        for fact, sk_cols in fact_tables.items():
            if not self.assert_table_exists(fact, fact):
                continue

            columns = self._get_columns(fact)
            missing = [c for c in sk_cols if c not in columns]
            if missing:
                self.warn(f"{fact}: configured key column(s) {missing} not found in schema, skipping.")
                continue

            for col in sk_cols:
                null_cnt = self._scalar(
                    f"SELECT COUNT(*) FROM {fact} WHERE {col} IS NULL"
                )
                if null_cnt > 0:
                    self.critical(f"{fact}: surrogate key '{col}' has {null_cnt:,} NULL(s).")
                else:
                    log.info(f"  OK  {fact:50s} {col} – no nulls")

    # ===================================================================
    # 4.  Numeric measures satisfy documented business rules
    # ===================================================================
    def check_business_rules(self):
        log.info("=" * 70)
        log.info("CHECK 4 – Numeric measures satisfy business rules")
        log.info("=" * 70)

        # ---- vw_annual_sales_kpis KPI validation --------------------------------
        kpi_view = f"{DB_GOLD}.vw_annual_sales_kpis"
        if self.assert_table_exists(kpi_view, "vw_annual_sales_kpis"):
            try:
                # 1. annual_revenue = SUM(l_extendedprice * (1 - l_discount))
                expected_revenue = self._scalar(
                    f"""
                    SELECT COALESCE(SUM(revenue), 0)
                    FROM {DB_GOLD}.fact_sales
                    WHERE YEAR(order_date) = 1992
                    """
                )

                # 2. active_customers = count(distinct customers with >=1 order
                #    in date range 1992-01-01 .. 1998-12-31)
                expected_active_customers = self._scalar(
                    f"""
                    SELECT COUNT(DISTINCT customer_key)
                    FROM {DB_GOLD}.fact_sales
                    WHERE order_date >= DATE('1992-01-01')
                      AND order_date <= DATE('1992-12-31')
                    """
                )

                # 3. orders = count(distinct o_orderkey) across all orders
                expected_orders = self._scalar(
                    f"""
                    SELECT COUNT(DISTINCT order_key)
                    FROM {DB_GOLD}.fact_sales
                    WHERE YEAR(order_date) = 1992
                    """
                )

                # 4. average_order_value = annual_revenue / distinct orders
                expected_avg_order_value = (expected_revenue / expected_orders) if expected_orders else 0

                # Read KPI values from the gold view
                kpi_rows = spark.sql(f"SELECT * FROM {kpi_view}").collect()
                if not kpi_rows:
                    self.critical(f"{kpi_view}: view returned 0 rows – cannot validate KPIs.")
                else:
                    kpi_row  = kpi_rows[0].asDict()
                    col_map = {c.lower(): c for c in kpi_row.keys()}

                    def _get_kpi(*candidates):
                        for c in candidates:
                            if c.lower() in col_map:
                                return kpi_row[col_map[c.lower()]]
                        return None

                    view_revenue       = _get_kpi("annual_revenue")
                    view_active_cust   = _get_kpi("active_customers")
                    view_orders        = _get_kpi("total_orders")
                    view_avg_order_val  = _get_kpi("avg_order_value")

                    # 1. Validate annual_revenue
                    if view_revenue is not None:
                        if abs(view_revenue - expected_revenue) > 0.01:
                            self.critical(
                                f"{kpi_view}: annual_revenue mismatch – view={view_revenue:,.2f} "
                                f"vs expected={expected_revenue:,.2f}"
                            )
                        else:
                            log.info(f"  OK  {kpi_view}: annual_revenue = {view_revenue:,.2f} (matches source)")
                    else:
                        self.warn(f"{kpi_view}: annual_revenue column not found in view.")

                    # 2. Validate active_customers
                    if view_active_cust is not None:
                        if view_active_cust != expected_active_customers:
                            self.critical(
                                f"{kpi_view}: active_customers mismatch – view={view_active_cust:,} "
                                f"vs expected={expected_active_customers:,}"
                            )
                        else:
                            log.info(f"  OK  {kpi_view}: active_customers = {view_active_cust:,} (matches source)")
                    else:
                        self.warn(f"{kpi_view}: active_customers column not found in view.")

                    # 3. Validate orders
                    if view_orders is not None:
                        if view_orders != expected_orders:
                            self.critical(
                                f"{kpi_view}: orders mismatch – view={view_orders:,} "
                                f"vs expected={expected_orders:,}"
                            )
                        else:
                            log.info(f"  OK  {kpi_view}: orders = {view_orders:,} (matches source)")
                    else:
                        self.warn(f"{kpi_view}: orders column not found in view.")

                    # 4. Validate average_order_value
                    if view_avg_order_val is not None:
                        if abs(view_avg_order_val - expected_avg_order_value) > 0.01:
                            self.critical(
                                f"{kpi_view}: average_order_value mismatch – view={view_avg_order_val:,.2f} "
                                f"vs expected={expected_avg_order_value:,.2f}"
                            )
                        else:
                            log.info(f"  OK  {kpi_view}: average_order_value = {view_avg_order_val:,.2f} (matches source)")
                    else:
                        self.warn(f"{kpi_view}: average_order_value column not found in view.")

                log.info(f"  OK  {kpi_view}: KPI business-rule checks completed")
            except Exception as e:
                self.warn(f"{kpi_view}: could not validate KPIs – {e}")

    # ===================================================================
    # 5.  Gold revenue and row counts reconcile with Silver
    # ===================================================================
    def reconcile_gold_to_silver(self):
        log.info("=" * 70)
        log.info("CHECK 5 – Gold revenue & row counts reconcile with Silver")
        log.info("=" * 70)

        TOL_PCT = 0.01   # 0.01 % tolerance for floating-point rounding

        # ---- row-count reconciliation ----------------------------------------------
        if not (self.assert_table_exists(f"{DB_SILVER}.lineitem") and self.assert_table_exists(f"{DB_GOLD}.fact_sales")):
            self.warn("Cannot reconcile row counts: Silver.lineitem or Gold.fact_sales not found.")
        else:
            silver_sales_rc = self._row_count(f"{DB_SILVER}.lineitem")
            gold_fact_rc    = self._row_count(f"{DB_GOLD}.fact_sales")

            if silver_sales_rc != gold_fact_rc:
                self.critical(
                    f"Row-count mismatch: Silver.lineitem={silver_sales_rc:,} "
                    f"vs Gold.fact_sales={gold_fact_rc:,} "
                    f"(delta={silver_sales_rc - gold_fact_rc:,})"
                )
            else:
                log.info(f"  OK  Row count reconciled: {silver_sales_rc:,} = {gold_fact_rc:,}")

        # ---- revenue reconciliation -----------------------------------------------
        if self.assert_table_exists(f"{DB_SILVER}.lineitem") and self.assert_table_exists(f"{DB_GOLD}.fact_sales"):
            silver_rev = self._scalar(
                f"""
                SELECT COALESCE(SUM(sli.l_extendedprice * (1 - sli.l_discount)), 0)
                FROM {DB_SILVER}.lineitem sli
                """
            )
            gold_rev = self._scalar(
                f"""
                SELECT COALESCE(SUM(fs.revenue), 0)
                FROM {DB_GOLD}.fact_sales fs
                """
            )

            if silver_rev == 0 and gold_rev == 0:
                log.info("  OK  Revenue = 0 on both sides.")
            else:
                diff_pct = abs(silver_rev - gold_rev) / max(abs(silver_rev), 1) * 100
                if diff_pct > TOL_PCT:
                    self.critical(
                        f"Revenue mismatch: Silver={silver_rev:,.2f} "
                        f"vs Gold={gold_rev:,.2f} "
                        f"(diff%={diff_pct:.6f}%)"
                    )
                else:
                    log.info(
                        f"  OK  Revenue reconciled within tolerance "
                        f"(Silver={silver_rev:,.2f}, Gold={gold_rev:,.2f}, diff%={diff_pct:.6f}%)"
                    )
        else:
            self.warn("Cannot reconcile revenue: Silver.lineitem or Gold.fact_sales not found.")

        # ---- line-item row-count reconciliation (if fact is at line-item grain) ---
        if self.assert_table_exists(f"{DB_SILVER}.lineitem") and self.assert_table_exists(f"{DB_GOLD}.fact_sales"):
            silver_li_rc = self._row_count(f"{DB_SILVER}.lineitem")
            gold_li_rc   = self._row_count(f"{DB_GOLD}.fact_sales")
            if silver_li_rc != gold_li_rc:
                self.warn(
                    f"Silver.lineitem rows ({silver_li_rc:,}) "
                    f"!= Gold.fact_sales rows ({gold_li_rc:,}). "
                    f"If fact_sales is at order grain this is expected; verify grain."
                )

    # ===================================================================
    # 6.  Rejected-record counts are reported
    # ===================================================================
    def report_rejected_records(self):
        log.info("=" * 70)
        log.info("CHECK 6 – Rejected-record counts reported")
        log.info("=" * 70)

        reject_tables = [
            f"{DB_BRONZE}.rejections",
            f"{DB_SILVER}.rejections",
            f"{DB_BRONZE}.sales_rejections",
            f"{DB_BRONZE}.sales_line_item_rejections",
            f"{DB_BRONZE}.customer_rejections",
            f"{DB_BRONZE}.product_rejections",
            f"{DB_BRONZE}.store_rejections",
            f"{DB_BRONZE}.promotion_rejections",
            f"{DB_BRONZE}.inventory_rejections",
            f"{DB_BRONZE}.employee_rejections",
        ]

        total_rejected = 0
        found_any = False
        for tbl in reject_tables:
            try:
                cnt = self._row_count(tbl)
                found_any = True
                total_rejected += cnt
                if cnt > 0:
                    log.info(f"  REJECT  {tbl:55s} rejected={cnt:,}")
                    # Sample a few reasons if the column exists
                    try:
                        sample = spark.sql(
                            f"""
                            SELECT rejection_reason, COUNT(*) AS cnt
                            FROM {tbl}
                            GROUP BY rejection_reason
                            ORDER BY cnt DESC
                            LIMIT 5
                            """
                        ).collect()
                        for r in sample:
                            log.info(f"          reason='{r['rejection_reason']}'  count={r['cnt']:,}")
                    except Exception:
                        pass  # column may not exist on all reject tables
                else:
                    log.info(f"  OK      {tbl:55s} rejected=0")
            except Exception:
                pass  # table not present – skip silently

        if not found_any:
            self.warn("No rejection/quarantine tables found – cannot report rejected-record counts.")
        else:
            log.info(f"  TOTAL rejected records across all layers: {total_rejected:,}")
            # Store the count as a SQL temp view for downstream visibility
            spark.sql(
                f"""
                CREATE OR REPLACE TEMP VIEW dq_rejected_record_summary AS
                SELECT {total_rejected} AS total_rejected_records,
                       current_timestamp() AS checked_at
                """
            )

    # ===================================================================
    # 7.  Aggregate results & fail the pipeline if any critical check failed
    # ===================================================================
    def finalize(self):
        log.info("=" * 70)
        log.info("DATA-QUALITY SUMMARY")
        log.info("=" * 70)
        log.info(f"  Critical failures : {len(self.critical_failures)}")
        log.info(f"  Warnings          : {len(self.warnings)}")

        if self.critical_failures:
            log.error("----- CRITICAL FAILURES -----")
            for i, msg in enumerate(self.critical_failures, 1):
                log.error(f"  {i}. {msg}")
            log.error("----- PIPELINE WILL BE STOPPED -----")
            summary = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(self.critical_failures))
            raise DQFailure(
                f"{len(self.critical_failures)} critical data-quality check(s) failed:\n{summary}"
            )
        else:
            log.info("ALL CRITICAL DATA-QUALITY CHECKS PASSED ✅")

        if self.warnings:
            log.warning("----- WARNINGS (non-blocking) -----")
            for i, msg in enumerate(self.warnings, 1):
                log.warning(f"  {i}. {msg}")


# ---------------------------------------------------------------------
# Run all checks
# ---------------------------------------------------------------------
runner = DataQualityRunner()
runner.check_source_ingestion()       # 1 – all eight sources ingested
runner.check_primary_keys()           # 2 – PKs populated & unique
runner.check_fact_surrogate_keys()    # 3 – no null surrogate keys in fact
runner.check_business_rules()         # 4 – numeric measures satisfy rules
runner.reconcile_gold_to_silver()     # 5 – Gold reconciles with Silver
runner.report_rejected_records()      # 6 – rejected-record counts reported
runner.finalize()                     # 7 – raise if any critical failure